In [4]:
!git clone https://github.com/lorenzo-stacchio/Deep-Armocromia.git

fatal: destination path 'Deep-Armocromia' already exists and is not an empty directory.


In [1]:
import io
import ast
import time
import base64
import re
import logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from datasets import load_dataset
from openai import OpenAI
from tqdm import tqdm
from PIL import Image
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# ====================================================
# 로깅 설정
# ====================================================
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("benchmark_eval.log", encoding="utf-8")
    ]
)
logger = logging.getLogger(__name__)

# ====================================================
# 1. 환경 설정
# ====================================================
RUNPOD_BASE_URL = "https://djufqa63ka3tur-8000.proxy.runpod.net/v1"
API_KEY         = "qwenbench123"
MODEL_NAME      = "Qwen/Qwen2.5-VL-7B-Instruct"

BASE_DIR       = Path("./Deep-Armocromia/dataset")
ARMOCROMIA_CSV = BASE_DIR / "annotations.csv"

MAX_WORKERS  = 6
DIAG_PRINT_N = 10  # 태스크별 처음 N개 GT/Pred 콘솔 출력

# ====================================================
# 태스크별 max_tokens
# (inspect 결과 기반 — 실제 존재하는 태스크 전체 포함)
# ====================================================
TASK_MAX_TOKENS = {
    "Color Recognition":  50,
    "Color Extraction":   80,
    "Color Proportion":  100,
    "Color Comparison":  100,
    "Color Robustness":   50,   # ← 수정: "Robustness" → "Color Robustness"
    "Color Blindness":    50,   # ← 신규
    "Color Counting":     50,   # ← 신규
    "Color Illusion":     50,   # ← 신규
    "Color Mimicry":      80,   # ← 신규
    "Object Counting":    50,   # ← 신규
    "Object Recognition": 50,   # ← 신규
    "Armocromia":         20,
}
DEFAULT_MAX_TOKENS = 100

# ====================================================
# 평가할 ColorBench 태스크 목록
# (inspect 결과에서 실제 존재하는 태스크만 포함)
# ====================================================
TARGET_TASKS = [
    "Color Recognition",
    "Color Extraction",
    "Color Proportion",
    "Color Comparison",
    "Color Robustness",   # ← 수정
    "Color Blindness",    # ← 신규
    "Color Counting",     # ← 신규
    "Color Illusion",     # ← 신규
    "Color Mimicry",      # ← 신규
    "Object Counting",    # ← 신규
    "Object Recognition", # ← 신규
]

client = OpenAI(base_url=RUNPOD_BASE_URL, api_key=API_KEY)


# ====================================================
# 2. 유틸리티 함수
# ====================================================
def encode_image(img_input) -> str:
    """이미지 경로 또는 PIL 객체를 Base64로 인코딩"""
    if isinstance(img_input, (str, Path)):
        img = Image.open(img_input)
    else:
        img = img_input
    if img.mode != "RGB":
        img = img.convert("RGB")
    max_size = 512
    if img.width > max_size or img.height > max_size:
        img.thumbnail((max_size, max_size), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()


@retry(
    retry=retry_if_exception_type(Exception),
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    reraise=False
)
def _call_api(img_b64: str, question: str, max_tokens: int) -> str:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text",      "text": question},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}},
            ]
        }],
        max_tokens=max_tokens,
        temperature=0,
        timeout=180,
    )
    return response.choices[0].message.content.strip()


def ask_vlm(question: str, img_input, task_name: str = "default") -> tuple[str, float]:
    max_tokens = TASK_MAX_TOKENS.get(task_name, DEFAULT_MAX_TOKENS)
    try:
        img_b64  = encode_image(img_input)
        start    = time.time()
        result   = _call_api(img_b64, question, max_tokens)
        elapsed  = time.time() - start
        return result, elapsed
    except Exception as e:
        logger.warning(f"[API 최종 실패] task={task_name}, error={e}")
        return None, 0.0


# ====================================================
# 3. choices 파싱 헬퍼
# ====================================================
def parse_choices(choices) -> list:
    if isinstance(choices, list):
        return choices
    if isinstance(choices, str):
        try:
            parsed = ast.literal_eval(choices)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
    return []


# ====================================================
# 4. MCQ 정답 추출
# ====================================================
def resolve_answer(row: dict) -> str:
    """
    answer 필드가 '(A)'~'(F)' 레이블이면 choices에서 실제 텍스트 추출.
    수정: A~D → A~F (5지선다·6지선다 대응)
    """
    answer  = str(row.get("answer", "")).strip()
    answer  = re.sub(r"[()]", "", answer).strip()  # (D) → D

    choices = parse_choices(row.get("choices", []))

    # A~F 레이블 처리 (수정: 기존 A~D → A~F)
    if re.match(r"^[A-Fa-f]$", answer):
        idx = ord(answer.upper()) - ord("A")
        if idx < len(choices):
            return str(choices[idx]).strip().lower()
        else:
            # 범위 초과 — 해당 샘플 제외
            logger.warning(f"answer '{answer}' 범위 초과 (choices={len(choices)}개) — 스킵")
            return ""

    return answer.lower()


# ====================================================
# 5. Pred 정규화
# ====================================================
def resolve_pred(pred: str, choices: list) -> str:
    """
    모델 출력을 정답 비교 가능한 텍스트로 정규화.
    "D"        → choices[3]  (레이블만 답한 경우)
    "C) Green" → "green"     (레이블+텍스트인 경우)
    "Green"    → "green"     (이미 텍스트)
    수정: A~D → A~F
    """
    if not pred:
        return pred

    pred = pred.strip()

    # "C) Green" / "C. Green" 형식 → 뒤 텍스트 추출
    match = re.match(r"^[A-Fa-f][).]\s*(.+)", pred)
    if match:
        return match.group(1).strip().lower()

    # "C" 단독 레이블 → choices 텍스트 추출 (수정: A~F)
    if re.match(r"^[A-Fa-f]$", pred) and choices:
        idx = ord(pred.upper()) - ord("A")
        if idx < len(choices):
            return str(choices[idx]).strip().lower()

    return pred.lower()


# ====================================================
# 6. 정확도 평가 로직
# ====================================================
def normalize_answer(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_correct(pred: str, answer: str, task_name: str = "") -> bool:
    """
    다단계 매칭:
    1) 정확 매칭
    2) 부정 표현 감지
    3) 전체 pred 텍스트에서 단어 경계 매칭
    """
    if not pred or not answer:
        return False

    pred_norm = normalize_answer(pred)
    ans_norm  = normalize_answer(answer)

    if pred_norm == ans_norm:
        return True

    negation_patterns = [
        rf"\bnot\s+{re.escape(ans_norm)}\b",
        rf"\bisnt\s+{re.escape(ans_norm)}\b",
        rf"\bno\s+{re.escape(ans_norm)}\b",
        rf"\bnever\s+{re.escape(ans_norm)}\b",
    ]
    for pat in negation_patterns:
        if re.search(pat, pred_norm):
            return False

    if re.search(rf"\b{re.escape(ans_norm)}\b", pred_norm):
        return True

    return False


# ====================================================
# 7. 데이터셋 구조 진단
# ====================================================
def diagnose_dataset(ds, task_name: str, n: int = 3):
    samples = [row for row in ds if row.get("task") == task_name][:n]
    print(f"\n{'='*60}")
    print(f"[진단] Task: {task_name} — 샘플 {n}개")
    print(f"{'='*60}")
    for i, row in enumerate(samples):
        print(f"\n--- 샘플 {i+1} ---")
        print(f"  question : {str(row.get('question',''))[:80]}")
        print(f"  answer   : {row.get('answer','')}")
        print(f"  choices  : {row.get('choices','(없음)')}")
    print(f"{'='*60}\n")


# ====================================================
# 8. 프롬프트 생성
# ====================================================
def build_question(original_q: str, choices: list, task_name: str) -> str:
    """
    choices 수에 맞게 마지막 레이블을 동적으로 생성.
    수정: "A, B, C, or D" 하드코딩 → 실제 choices 수 기반
    """
    if choices:
        n     = len(choices)
        last  = chr(64 + n)  # 4→D, 5→E, 6→F
        lines = "\n".join([f"{chr(65+i)}) {c}" for i, c in enumerate(choices)])
        return (
            f"{original_q}\n\nChoices:\n{lines}\n\n"
            f"Answer with ONLY the letter (A to {last}). Do not explain."
        )
    else:
        # 자유응답
        return original_q + "\nAnswer in a few words only. Do not explain."


# ====================================================
# 9. ColorBench 종합 평가
# ====================================================
def eval_colorbench_full() -> dict:
    logger.info("🚀 [1/2] ColorBench 종합 평가 시작...")

    ds = load_dataset("umd-zhou-lab/ColorBench", split="test")

    # 진단 (첫 번째 태스크만)
    diagnose_dataset(ds, TARGET_TASKS[0], n=3)

    task_results = {}

    for task_name in TARGET_TASKS:
        task_ds = [row for row in ds if row.get("task") == task_name]
        if not task_ds:
            logger.warning(f"⚠️ Task [{task_name}] 데이터 없음, 스킵")
            continue

        logger.info(f"📊 [{task_name}] — {len(task_ds)}개 (Workers: {MAX_WORKERS})")

        correct    = 0
        valid      = 0
        failed_api = 0
        diag_count = 0

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {}
            for row in task_ds:
                original_q = row.get("question", "")
                choices    = parse_choices(row.get("choices", []))
                answer     = resolve_answer(row)

                # answer 없음(범위초과 등) → 스킵
                if not row.get("image") or not answer:
                    continue

                question = build_question(original_q, choices, task_name)
                future   = executor.submit(ask_vlm, question, row["image"], task_name)
                futures[future] = {
                    "question":   original_q,
                    "answer":     answer,
                    "raw_answer": row.get("answer", ""),
                    "choices":    choices,
                }

            for future in tqdm(as_completed(futures), total=len(futures), desc=task_name, leave=False):
                meta       = futures[future]
                pred, _    = future.result()

                if pred is None:
                    failed_api += 1
                    continue

                valid += 1

                # Pred도 텍스트로 변환 (핵심 수정)
                pred_resolved = resolve_pred(pred, meta["choices"])
                correct_flag  = is_correct(pred_resolved, meta["answer"], task_name)
                if correct_flag:
                    correct += 1

                if diag_count < DIAG_PRINT_N:
                    mark = "✅" if correct_flag else "❌"
                    print(
                        f"[DIAG][{task_name}] {mark} "
                        f"GT='{meta['answer']}' (raw='{meta['raw_answer']}') | "
                        f"Pred_raw='{pred[:40]}' | Pred_resolved='{pred_resolved[:40]}'"
                    )
                    diag_count += 1

                logger.debug(
                    f"Q: {meta['question'][:50]} | GT: {meta['answer']} | "
                    f"Pred: {pred_resolved[:30]} | {'✅' if correct_flag else '❌'}"
                )

        acc = (correct / valid * 100) if valid > 0 else 0.0
        task_results[task_name] = {
            "accuracy":   acc,
            "correct":    correct,
            "valid":      valid,
            "failed_api": failed_api,
            "total":      len(task_ds),
        }
        logger.info(f"  ✅ [{task_name}] {acc:.2f}% ({correct}/{valid}, API실패: {failed_api})")

    return task_results


# ====================================================
# 10. Deep-Armocromia 평가
# ====================================================
def eval_armocromia() -> dict:
    logger.info(f"\n🚀 [2/2] Deep-Armocromia 평가 시작 (Workers: {MAX_WORKERS})...")

    if not ARMOCROMIA_CSV.exists():
        logger.error(f"⚠️ CSV 없음: {ARMOCROMIA_CSV}")
        return {}

    df       = pd.read_csv(ARMOCROMIA_CSV)
    test_df  = df[df["partition"] == "test"].copy()

    if test_df.empty:
        logger.error("⚠️ test 파티션 없음")
        return {}

    logger.info(f"  test 샘플 수: {len(test_df)}개")

    SEASON_MAP = {
        "primavera": "spring",
        "estate":    "summer",
        "autunno":   "autumn",
        "inverno":   "winter",
    }

    PROMPT = (
        "You are an expert in Armocromia personal color analysis. "
        "Carefully examine the person's skin tone, hair color, and eye color in the image. "
        "Classify their seasonal palette into exactly one of: Spring, Summer, Autumn, Winter. "
        "Respond with ONLY the single season word. Do not explain."
    )

    correct       = 0
    valid         = 0
    failed_api    = 0
    class_correct = {s: 0 for s in SEASON_MAP.values()}
    class_total   = {s: 0 for s in SEASON_MAP.values()}
    diag_count    = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {}
        for _, row in test_df.iterrows():
            img_path = BASE_DIR / str(row["path_rgb_original"])
            gt       = SEASON_MAP.get(str(row["class"]).strip().lower(), "")

            if not img_path.exists() or not gt:
                continue

            future          = executor.submit(ask_vlm, PROMPT, img_path, "Armocromia")
            futures[future] = gt

        for future in tqdm(as_completed(futures), total=len(futures), desc="Armocromia"):
            gt      = futures[future]
            pred, _ = future.result()

            if pred is None:
                failed_api += 1
                continue

            valid += 1
            class_total[gt] = class_total.get(gt, 0) + 1

            correct_flag = is_correct(pred, gt, "Armocromia")
            if correct_flag:
                correct += 1
                class_correct[gt] = class_correct.get(gt, 0) + 1

            if diag_count < DIAG_PRINT_N:
                mark = "✅" if correct_flag else "❌"
                print(f"[DIAG][Armocromia] {mark} GT='{gt}' | Pred='{pred[:60]}'")
                diag_count += 1

            logger.debug(f"GT: {gt} | Pred: {pred} | {'✅' if correct_flag else '❌'}")

    overall_acc = (correct / valid * 100) if valid > 0 else 0.0
    per_class   = {
        s: (class_correct.get(s, 0) / class_total.get(s, 1) * 100)
        if class_total.get(s, 0) > 0 else 0.0
        for s in SEASON_MAP.values()
    }

    return {
        "accuracy":   overall_acc,
        "correct":    correct,
        "valid":      valid,
        "failed_api": failed_api,
        "per_class":  per_class,
    }


# ====================================================
# 11. 리포트 출력
# ====================================================
def print_report(cb_scores: dict, armo_result: dict):
    print("\n" + "=" * 65)
    print("📊  Qwen2.5-VL 프로젝트 통합 벤치마크 리포트")
    print("=" * 65)

    print("\n[ ColorBench ]")
    print(f"  {'Task':<24} {'Accuracy':>8}  {'Correct/Valid':>14}  {'API Fail':>8}")
    print("  " + "-" * 62)
    cb_avg = []
    for task, info in cb_scores.items():
        print(
            f"  {task:<24} {info['accuracy']:>7.2f}%"
            f"  {info['correct']:>5}/{info['valid']:<8}"
            f"  {info['failed_api']:>8}"
        )
        cb_avg.append(info["accuracy"])
    if cb_avg:
        print(f"\n  {'평균':<24} {sum(cb_avg)/len(cb_avg):>7.2f}%")

    print("\n[ Deep-Armocromia ]")
    if armo_result:
        print(
            f"  전체 정확도: {armo_result['accuracy']:.2f}%  "
            f"({armo_result['correct']}/{armo_result['valid']}, "
            f"API 실패: {armo_result['failed_api']})"
        )
        print("  클래스별:")
        for season, acc in armo_result.get("per_class", {}).items():
            print(f"    {season:<8}: {acc:.2f}%")
    else:
        print("  ⚠️ 결과 없음")

    print("\n" + "=" * 65)


# ====================================================
# 12. 실행
# ====================================================
if __name__ == "__main__":
    start_time  = time.time()
    cb_scores   = eval_colorbench_full()
    armo_result = eval_armocromia()
    print_report(cb_scores, armo_result)
    elapsed = time.time() - start_time
    print(f"⏱️  총 소요 시간: {int(elapsed//60)}분 {int(elapsed%60)}초")

2026-05-15 18:58:46,655 [INFO] 🚀 [1/2] ColorBench 종합 평가 시작...
2026-05-15 18:58:46,672 [DEBUG] connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-05-15 18:58:46,697 [DEBUG] connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000002977FD93380>
2026-05-15 18:58:46,699 [DEBUG] start_tls.started ssl_context=<ssl.SSLContext object at 0x000002977FDC9310> server_hostname='huggingface.co' timeout=10
2026-05-15 18:58:46,719 [DEBUG] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000002977FD25F90>
2026-05-15 18:58:46,720 [DEBUG] send_request_headers.started request=<Request [b'HEAD']>
2026-05-15 18:58:46,722 [DEBUG] send_request_headers.complete
2026-05-15 18:58:46,723 [DEBUG] send_request_body.started request=<Request [b'HEAD']>
2026-05-15 18:58:46,725 [DEBUG] send_request_body.complete
2026-05-15 18:58:46,726 [DEBUG] receive_response_headers.started request=<Request [b'HEAD']>
20

KeyboardInterrupt: 